<a href="https://colab.research.google.com/github/Farhan1Ahmed1Molla/universal-data-santizer/blob/main/case_studies_01_cafe_sales_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re

def sanitize_data(df):
    clean_df = df.copy()

    # LAYER 1: Standardize Column Headers
    clean_df.columns = (
        clean_df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[^\w\s]', '', regex=True)
        .str.replace(r'\s+', '_', regex=True)
    )

    # LAYER 2: Drop 100% Duplicate Rows
    initial_rows = len(clean_df)
    clean_df = clean_df.drop_duplicates().reset_index(drop=True)
    dupes_removed = initial_rows - len(clean_df)

    # LAYER 3: Clean Text Columns & Convert Disguised Nulls
    fake_nulls = ['n/a', 'na', 'null', 'none', 'error', 'unknown', '-', '--', '?', 'tbd', '']

    for col in clean_df.select_dtypes(include='object').columns:
        clean_df[col] = clean_df[col].astype(str).str.strip()
        clean_df[col] = clean_df[col].apply(
            lambda x: np.nan if x.lower() in fake_nulls or x == 'nan' else x
        )

    # LAYERS 4 & 5: Auto-Date Parsing & Smart Numeric Scrubbing
    for col in clean_df.select_dtypes(include='object').columns:
        non_null_count = clean_df[col].notna().sum()
        if non_null_count == 0:
            continue

        # Layer 4: Dates
        if 'date' in col:
            clean_df[col] = pd.to_datetime(clean_df[col], errors='coerce', format='mixed')
            clean_df = clean_df.sort_values(by=col, ascending=True).reset_index(drop=True)
            continue

        # Layer 5: Numbers
        temp_num = clean_df[col].str.replace(r'[$€£₹,%]', '', regex=True).str.strip()
        converted_num = pd.to_numeric(temp_num, errors='coerce')

        if (converted_num.notna().sum() / non_null_count) >= 0.70:
            clean_df[col] = converted_num

    # FINAL OUTPUT (Must stay at the very bottom of the function)
    print(f"Headers standardized | Duplicate rows removed: {dupes_removed}")
    return clean_df

In [ ]:
import kagglehub
import os

# Download Dataset #1 directly from Kaggle
path = kagglehub.dataset_download("ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training")
csv_path = os.path.join(path, os.listdir(path)[0])

# Load raw dataset and inspect messy data types
raw_cafe_df = pd.read_csv(csv_path)
print("--- BEFORE SANITIZING ---")
print(raw_cafe_df.dtypes)

# Run your Universal Data Sanitizer
print("\n--- RUNNING SANITIZER ---")
clean_cafe_df = sanitize_data(raw_cafe_df)

print("\n--- AFTER SANITIZING ---")
print(clean_cafe_df.dtypes)

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.
--- BEFORE SANITIZING ---
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

--- RUNNING SANITIZER ---
Headers standardized | Duplicate rows removed: 0

--- AFTER SANITIZING ---
transaction_id              object
item                        object
quantity                   float64
price_per_unit             float64
total_spent                float64
payment_method              object
location                    object
transaction_date    datetime64[ns]
dtype: object


In [ ]:
clean_cafe_df.isna().sum()

,0
transaction_id,0
item,969
quantity,479
price_per_unit,533
total_spent,502
payment_method,3178
location,3961
transaction_date,460


In [ ]:
unique_ids = clean_cafe_df['transaction_id'].nunique()
total_rows = len(clean_cafe_df)
print(f"Unique IDs: {unique_ids} / {total_rows}")

valid_format = clean_cafe_df['transaction_id'].str.match(r'^TXN_\d{7}$').all()
print(f"All Ids match 'TXN_XXXXXXX' format: {valid_format}")

assert unique_ids == total_rows, "Duplicate Transaction IDs detected!"
assert valid_format, "Malformed Transaction ID strings detected!"
print("column 1 (transaction_id) verified clean - no changes needed.")

Unique IDs: 10000 / 10000
All Ids match 'TXN_XXXXXXX' format: True
column 1 (transaction_id) verified clean - no changes needed.


In [ ]:
print("--- Unique Items Count---")
print(clean_cafe_df['item'].value_counts(dropna=False))

print("\n--- Menu Price Map---")
print(clean_cafe_df.groupby('item')['price_per_unit'].value_counts())

--- Unique Items Count---
item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
NaN          969
Name: count, dtype: int64

--- Menu Price Map---
item      price_per_unit
Cake      3.0               1085
Coffee    2.0               1108
Cookie    1.0               1026
Juice     3.0               1110
Salad     5.0               1082
Sandwich  4.0               1082
Smoothie  4.0               1036
Tea       1.5               1023
Name: count, dtype: int64


In [ ]:
unique_price_to_item = {
    1.0: 'Cookie',
    1.5: 'Tea',
    2.0: 'Coffee',
    5.0: 'Salad'
}

before_nulls = clean_cafe_df['item'].isna().sum()
clean_cafe_df['item'] = clean_cafe_df['item'].fillna(
    clean_cafe_df['price_per_unit'].map(unique_price_to_item))
after_nulls = clean_cafe_df['item'].isna().sum()

print(f"Recovered {before_nulls - after_nulls} missing items using unique prices.")
print(f"Remaining NaN in 'item': {after_nulls}")

expected_items = {'Juice', 'Coffee', 'Salad', 'Cake', 'Sandwich', 'Smoothie', 'Cookie', 'Tea'}
actual_items = set(clean_cafe_df['item'].dropna().unique())
assert actual_items == expected_items, f"Unexpected items found: {actual_items - expected_items}"
print("column 2 (item) verified - only valid items present.")


Recovered 468 missing items using unique prices.
Remaining NaN in 'item': 501
column 2 (item) verified - only valid items present.


In [ ]:
print("--- Quantity distribution---")
print(clean_cafe_df['quantity'].value_counts(dropna=False).sort_index())

before_nulls = clean_cafe_df['quantity'].isna().sum()
clean_cafe_df['quantity'] = clean_cafe_df['quantity'].fillna(
    clean_cafe_df['total_spent'] / clean_cafe_df['price_per_unit']
)
after_qty_nulls = clean_cafe_df['quantity'].isna().sum()

print(f"\nRecovered {before_nulls - after_qty_nulls} missing quantities via math.")
print(f"Remaining NaN in 'quantity': {after_qty_nulls}")

valid_qty = clean_cafe_df['quantity'].dropna()
assert (valid_qty > 0).all(), "Zero or negative quantity detected!"
assert (valid_qty % 1 == 0).all(), "Non-integer quantity detected!"
print("column 3 (quantity) verified - all values are positive integers.")




--- Quantity distribution---
quantity
1.0    1822
2.0    1974
3.0    1849
4.0    1863
5.0    2013
NaN     479
Name: count, dtype: int64

Recovered 441 missing quantities via math.
Remaining NaN in 'quantity': 38
column 3 (quantity) verified - all values are positive integers.


In [ ]:
print("--- PRICE PER UNIT DISTRIBUTION ---")
print(clean_cafe_df['price_per_unit'].value_counts(dropna=False).sort_index())

before_price_nulls = clean_cafe_df['price_per_unit'].isna().sum()

item_to_price = {
    'Cookie': 1.0,
    'Tea': 1.5,
    'Coffee': 2.0,
    'Cake': 3.0,
    'Juice': 3.0,
    'Sandwich': 4.0,
    'Smoothie': 4.0,
    'Salad': 5.0
}
clean_cafe_df['price_per_unit'] = clean_cafe_df['price_per_unit'].fillna(
    clean_cafe_df['item'].map(item_to_price)
)
clean_cafe_df['price_per_unit'] = clean_cafe_df['price_per_unit'].fillna(
    clean_cafe_df['total_spent'] / clean_cafe_df['quantity']
)

after_price_nulls = clean_cafe_df['price_per_unit'].isna().sum()
print(f"\nRecovered {before_price_nulls - after_price_nulls} missing prices.")
print(f"Remaining NaN in 'price_per_unit': {after_price_nulls}")

expected_prices = {1.0, 1.5, 2.0, 3.0, 4.0, 5.0}
actual_prices = set(clean_cafe_df['price_per_unit'].dropna().unique())
assert actual_prices == expected_prices, f"Unexpected price found: {actual_prices - expected_prices}"
print("Column 4 (price_per_unit) verified — all values match official menu prices.")

--- PRICE PER UNIT DISTRIBUTION ---
price_per_unit
1.0    1143
1.5    1133
2.0    1227
3.0    2429
4.0    2331
5.0    1204
NaN     533
Name: count, dtype: int64

Recovered 527 missing prices.
Remaining NaN in 'price_per_unit': 6
Column 4 (price_per_unit) verified — all values match official menu prices.


In [ ]:
before_total_nulls = clean_cafe_df['total_spent'].isna().sum()
print(f"Missing 'total_spent' before recovery: {before_total_nulls}")

clean_cafe_df['total_spent'] = clean_cafe_df['total_spent'].fillna(
    clean_cafe_df['quantity'] * clean_cafe_df['price_per_unit']
)

after_total_nulls = clean_cafe_df['total_spent'].isna().sum()
print(f"Recovered {before_total_nulls - after_total_nulls} missing totals via math.")
print(f"Remaining NaN in 'total_spent': {after_total_nulls}")

valid_totals = clean_cafe_df['total_spent'].dropna()
assert (valid_totals > 0).all(), "Zero or negative total_spent detected!"

complete_math = clean_cafe_df.dropna(subset=['quantity', 'price_per_unit', 'total_spent'])
assert np.allclose(
    complete_math['quantity'] * complete_math['price_per_unit'],
    complete_math['total_spent']
), "Math mismatch found between Quantity * Price and Total Spent!"

print("Column 5 (total_spent) verified — all math balances.")

Missing 'total_spent' before recovery: 502
Recovered 479 missing totals via math.
Remaining NaN in 'total_spent': 23
Column 5 (total_spent) verified — all math balances.


In [ ]:
print("--- PAYMENT METHOD DISTRIBUTION (BEFORE) ---")
print(clean_cafe_df['payment_method'].value_counts(dropna=False))

missing_pm = clean_cafe_df['payment_method'].isna().sum()
clean_cafe_df['payment_method'] = clean_cafe_df['payment_method'].fillna('Unspecified')

print(f"\nTagged {missing_pm} missing payment methods as 'Unspecified'.")

# 3. Manual Reviewer Assertions: Zero nulls left and only valid categories exist
assert clean_cafe_df['payment_method'].isna().sum() == 0, "Nulls still remain in payment_method!"
expected_methods = {'Cash', 'Credit Card', 'Digital Wallet', 'Unspecified'}
actual_methods = set(clean_cafe_df['payment_method'].unique())
assert actual_methods.issubset(expected_methods), f"Unexpected payment method: {actual_methods - expected_methods}"

print("Column 6 (payment_method) verified — all categories standardized.")

--- PAYMENT METHOD DISTRIBUTION (BEFORE) ---
payment_method
NaN               3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

Tagged 3178 missing payment methods as 'Unspecified'.
Column 6 (payment_method) verified — all categories standardized.


In [ ]:

print("--- LOCATION DISTRIBUTION (BEFORE) ---")
print(clean_cafe_df['location'].value_counts(dropna=False))


missing_loc = clean_cafe_df['location'].isna().sum()
clean_cafe_df['location'] = clean_cafe_df['location'].fillna('Unspecified')

print(f"\nTagged {missing_loc} missing locations as 'Unspecified'.")

assert clean_cafe_df['location'].isna().sum() == 0, "Nulls still remain in location!"
expected_locations = {'In-store', 'Takeaway', 'Unspecified'}
actual_locations = set(clean_cafe_df['location'].unique())
assert actual_locations.issubset(expected_locations), f"Unexpected location: {actual_locations - expected_locations}"

print("Column 7 (location) verified — all categories standardized.")

--- LOCATION DISTRIBUTION (BEFORE) ---
location
NaN         3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64

Tagged 3961 missing locations as 'Unspecified'.
Column 7 (location) verified — all categories standardized.


In [ ]:

missing_dates = clean_cafe_df['transaction_date'].isna().sum()
min_date = clean_cafe_df['transaction_date'].min()
max_date = clean_cafe_df['transaction_date'].max()

print(f"Data type: {clean_cafe_df['transaction_date'].dtype}")
print(f"Missing dates (NaT): {missing_dates}")
print(f"Date range: {min_date.date()} to {max_date.date()}")

clean_cafe_df = clean_cafe_df.sort_values(
    by='transaction_date', ascending=True, na_position='last'
).reset_index(drop=True)


valid_dates = clean_cafe_df['transaction_date'].dropna()
assert pd.api.types.is_datetime64_any_dtype(clean_cafe_df['transaction_date']), "Column is not datetime!"
assert (valid_dates >= pd.Timestamp('2020-01-01')).all() and (valid_dates <= pd.Timestamp('2026-11-18')).all(), "Impossible date detected!"
assert valid_dates.is_monotonic_increasing, "Dates are not sorted chronologically!"

print("Column 8 (transaction_date) verified — sorted chronologically with valid dates.")

Data type: datetime64[ns]
Missing dates (NaT): 460
Date range: 2023-01-01 to 2023-12-31
Column 8 (transaction_date) verified — sorted chronologically with valid dates.


In [ ]:

unique_price_to_item = {
    1.0: 'Cookie',
    1.5: 'Tea',
    2.0: 'Coffee',
    5.0: 'Salad'
}
before_sweep = clean_cafe_df['item'].isna().sum()
clean_cafe_df['item'] = clean_cafe_df['item'].fillna(
    clean_cafe_df['price_per_unit'].map(unique_price_to_item)
)
rescued_wave2 = before_sweep - clean_cafe_df['item'].isna().sum()


remaining_unspec = clean_cafe_df['item'].isna().sum()
clean_cafe_df['item'] = clean_cafe_df['item'].fillna('Unspecified')

print(f"Second-wave rescue recovered {rescued_wave2} more items.")
print(f"Tagged {remaining_unspec} ambiguous/unknown items as 'Unspecified'.")


cafe_audit = pd.DataFrame({
    'Raw_Type': raw_cafe_df.dtypes.values,
    'Clean_Type': clean_cafe_df.dtypes.values,
    'Raw_Nulls': raw_cafe_df.isna().sum().values,
    'Final_Nulls': clean_cafe_df.isna().sum().values
}, index=clean_cafe_df.columns)

print("\n--- CAFE SALES: FINAL BEFORE VS. AFTER AUDIT ---")
print(cafe_audit)
print("\n--- CLEANED SAMPLE (FIRST 5 ROWS) ---")
display(clean_cafe_df.head(5))


assert len(clean_cafe_df) == 10000, "Row count changed unexpectedly!"
assert clean_cafe_df['item'].isna().sum() == 0, "Nulls still exist in 'item'!"
print("\nCafe Sales dataset 100% finalized and verified!")

Second-wave rescue recovered 0 more items.
Tagged 0 ambiguous/unknown items as 'Unspecified'.

--- CAFE SALES: FINAL BEFORE VS. AFTER AUDIT ---
                 Raw_Type      Clean_Type  Raw_Nulls  Final_Nulls
transaction_id     object          object          0            0
item               object          object        333            0
quantity           object         float64        138           38
price_per_unit     object         float64        179            6
total_spent        object         float64        173           23
payment_method     object          object       2579            0
location           object          object       3265            0
transaction_date   object  datetime64[ns]        159          460

--- CLEANED SAMPLE (FIRST 5 ROWS) ---


,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_4801947,Juice,1.0,3.0,3.0,Digital Wallet,Takeaway,2023-01-01
1,TXN_6234882,Cookie,4.0,1.0,4.0,Credit Card,Unspecified,2023-01-01
2,TXN_1581562,Coffee,2.0,2.0,4.0,Cash,In-store,2023-01-01
3,TXN_4292280,Cake,4.0,3.0,12.0,Cash,Takeaway,2023-01-01
4,TXN_6566716,Coffee,1.0,2.0,2.0,Credit Card,Unspecified,2023-01-01



Cafe Sales dataset 100% finalized and verified!
